In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits

sys.path.append(r'D:/CPD_MPIA/HPC_scripts/Source_codes/diskdictionary_r90')
import diskdictionaryr0_5 as disk

BASE     = r'D:/CPD_MPIA/HPC_scripts'
FITS_DIR = r'D:/CPD_MPIA/Standard_dy_SNR/Disk_Residual_Profile'
%matplotlib inline

## 1 — J1852 Rmax comparison (R90 vs rout)

In [ ]:
runs = {
    'inj_rev/J1852_gap0\n(rout, diskdict_pix)':
        os.path.join(BASE, 'J1852_gap0', 'mprofiles'),
    'J1852_gap0_r0_5\n(rout, diskdict_pix, rob0.5)':
        os.path.join(BASE, 'J1852_gap0_r0_5', 'mprofiles'),
    'inj_rev/r0_5/J1852_gap0\n(R90, diskdict_r90)':
        os.path.join(BASE, 'inj_rev', 'r0_5', 'J1852_gap0', 'mprofiles'),
}

rout_j = disk.disk['J1852']['rout']
R90_j  = disk.disk['J1852']['R90']
rgap_j = disk.disk['J1852']['rgap'][0]
wgap_j = disk.disk['J1852']['wgap'][0]
N_SAMPLE = 6

colors = ['tab:blue', 'tab:orange', 'tab:green']
fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 5), sharey=True)

for ax, (label, mdir), color in zip(axes, runs.items(), colors):
    files = sorted(glob.glob(os.path.join(mdir, '*_frank_profile_fit.txt')))
    if not files:
        ax.text(0.5, 0.5, 'no mprofiles\nfound', ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.set_title(label)
        continue
    for i, fpath in enumerate(files[:N_SAMPLE]):
        try:
            d = np.loadtxt(fpath)
            ax.plot(d[:,0], d[:,1]*1e-10, color=color, alpha=0.5, lw=1)
        except: pass
    ax.axvline(rout_j,          color='red',    ls='--', lw=1.2, label=f'rout={rout_j:.3f}"')
    ax.axvline(R90_j,           color='purple', ls='--', lw=1.2, label=f'R90={R90_j:.3f}"')
    ax.axvline(rgap_j - wgap_j, color='grey',   ls=':',  lw=1,   label='gap bounds')
    ax.axvline(rgap_j + wgap_j, color='grey',   ls=':',  lw=1)
    ax.axvline(2*rout_j,        color='red',    ls=':',  lw=1,   label=f'Rmax(rout)={2*rout_j:.2f}"')
    ax.axvline(2*R90_j,         color='purple', ls=':',  lw=1,   label=f'Rmax(R90)={2*R90_j:.2f}"')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('r [arcsec]')
    ax.legend(fontsize=7)
    ax.set_xlim(0, max(2*rout_j, 2*R90_j) * 1.1)

axes[0].set_ylabel('Brightness [scaled]')
plt.suptitle('J1852 gap0 — R90 vs rout Rmax', fontsize=11)
plt.tight_layout()
plt.show()

## 2 — R90 vs rout overview for all disks

In [ ]:
targets = sorted(disk.disk.keys())
rout_vals = [disk.disk[t]['rout'] for t in targets]
R90_vals  = [disk.disk[t].get('R90', np.nan) for t in targets]
ratios    = [r90/ro if not np.isnan(r90) else np.nan for r90, ro in zip(R90_vals, rout_vals)]

x = np.arange(len(targets))
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.bar(x - 0.2, rout_vals, 0.4, label='rout', color='tab:red', alpha=0.7)
ax.bar(x + 0.2, R90_vals,  0.4, label='R90',  color='tab:purple', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(targets, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('arcsec')
ax.legend()
ax.set_title('rout and R90 per disk')

ax = axes[1]
colors_bar = ['tab:red' if r > 2 else 'tab:blue' for r in ratios]
ax.bar(x, ratios, color=colors_bar, alpha=0.8)
ax.axhline(1, color='k', ls='--', lw=1)
ax.axhline(2, color='tab:red', ls=':', lw=1, label='R90/rout = 2 (warning)')
ax.set_xticks(x)
ax.set_xticklabels(targets, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('R90 / rout')
ax.set_title('R90/rout ratio — red bars are problematic (R90 >> rout)')
ax.legend()

plt.tight_layout()
plt.show()

print('\nDisk             rout      R90       ratio')
for t, ro, r90, rat in zip(targets, rout_vals, R90_vals, ratios):
    flag = '  *** large ratio' if rat > 2 else ''
    print(f'{t:<18} {ro:.3f}    {r90:.3f}    {rat:.2f}{flag}')

## 3 — Frank profiles for all available r0_5 gaps with R90 and rout marked

In [ ]:
r0_5_dir = os.path.join(BASE, 'inj_rev', 'r0_5')

gap_dirs = sorted([
    d for d in os.listdir(r0_5_dir)
    if os.path.isdir(os.path.join(r0_5_dir, d, 'mprofiles'))
    and glob.glob(os.path.join(r0_5_dir, d, 'mprofiles', '*_frank_profile_fit.txt'))
    and not d.startswith('J1852_gap0_test')
])

print(f'Found {len(gap_dirs)} gaps with mprofiles:', gap_dirs)

ncols = 4
nrows = max(1, int(np.ceil(len(gap_dirs) / ncols)))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.array(axes).flatten()

for ax, gdir in zip(axes, gap_dirs):
    parts = gdir.rsplit('_gap', 1)
    target, gap_ix = parts[0], int(parts[1])
    files = sorted(glob.glob(os.path.join(r0_5_dir, gdir, 'mprofiles', '*_frank_profile_fit.txt')))
    for fpath in files[:3]:
        try:
            d = np.loadtxt(fpath)
            ax.plot(d[:,0], d[:,1]*1e-10, color='tab:blue', alpha=0.5, lw=1)
        except: pass
    if target in disk.disk:
        ro  = disk.disk[target]['rout']
        r90 = disk.disk[target].get('R90', np.nan)
        ax.axvline(ro,    color='red',    ls='--', lw=1.2, label=f'rout={ro:.3f}"')
        ax.axvline(2*ro,  color='red',    ls=':',  lw=1)
        if not np.isnan(r90):
            ax.axvline(r90,   color='purple', ls='--', lw=1.2, label=f'R90={r90:.3f}"')
            ax.axvline(2*r90, color='purple', ls=':',  lw=1)
        if gap_ix < len(disk.disk[target]['rgap']):
            rg = disk.disk[target]['rgap'][gap_ix]
            wg = disk.disk[target]['wgap'][gap_ix]
            ax.axvspan(rg - wg, rg + wg, alpha=0.1, color='grey', label='gap')
        ax.legend(fontsize=6)
        xlim = max(2*ro, 2*r90 if not np.isnan(r90) else 0) * 1.1
        ax.set_xlim(0, max(xlim, 0.5))
    ax.set_title(gdir, fontsize=8)
    ax.set_xlabel('r [arcsec]', fontsize=7)

for ax in axes[len(gap_dirs):]:
    ax.set_visible(False)

plt.suptitle('Frank profiles (r0_5) — dashed=rout/R90, dotted=2×Rmax', fontsize=11)
plt.tight_layout()
plt.show()

## 4 — Robust=0.5 radial profiles (pre-computed by DiskResiduals_Median_SNR)
Columns: radius [arcsec], residual_I [Jy/beam], residual_std, clean_I [Jy/beam].
Blue = clean profile, orange = residual. rout (red dashed) and R90 (purple dashed) mark frank Rmax boundaries. Dotted = 2× (actual Rmax used).

In [ ]:
PROF_DIR = r'D:/CPD_MPIA/Median_SNR/Disk_Residual_Profile_Median_SNR'

targets_plot = sorted(disk.disk.keys())
ncols = 4
nrows = int(np.ceil(len(targets_plot) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.array(axes).flatten()

for ax, target in zip(axes, targets_plot):
    fpath = os.path.join(PROF_DIR, target, f'{target}_residual_radial_profile_robust0.5.txt')
    if not os.path.exists(fpath):
        ax.text(0.5, 0.5, 'no profile', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(target, fontsize=8)
        continue
    prof = np.loadtxt(fpath, comments='#')
    r       = prof[:, 0]
    I_resid = prof[:, 1] * 1e6
    I_clean = prof[:, 3] * 1e6
    ax.plot(r, I_clean, color='tab:blue',   lw=1.2, label='clean')
    ax.plot(r, I_resid, color='tab:orange', lw=0.8, alpha=0.7, label='residual')
    ax.axhline(0, color='k', ls=':', lw=0.5)
    d   = disk.disk[target]
    ro  = d['rout']
    r90 = d.get('R90', np.nan)
    ax.axvline(ro,   color='red',    ls='--', lw=1.5, label=f'rout={ro:.3f}"')
    ax.axvline(2*ro, color='red',    ls=':',  lw=1,   label=f'2×rout={2*ro:.2f}"')
    if not np.isnan(r90):
        ax.axvline(r90,   color='purple', ls='--', lw=1.5, label=f'R90={r90:.3f}"')
        ax.axvline(2*r90, color='purple', ls=':',  lw=1,   label=f'2×R90={2*r90:.2f}"')
    for rg, wg in zip(d['rgap'], d['wgap']):
        ax.axvspan(rg - wg, rg + wg, alpha=0.12, color='grey')
    xlim = max(2*ro, 2*r90 if not np.isnan(r90) else 0) * 1.15
    ax.set_xlim(0, max(xlim, 0.5))
    ax.set_title(target, fontsize=9)
    ax.set_xlabel('r [arcsec]', fontsize=7)
    ax.set_ylabel('µJy/beam', fontsize=7)
    ax.legend(fontsize=6, loc='upper right')

for ax in axes[len(targets_plot):]:
    ax.set_visible(False)

plt.suptitle('Robust=0.5 radial profiles — dashed=rout/R90, dotted=2×(frank Rmax), grey=gap', fontsize=10)
plt.tight_layout()
plt.show()